# YouTube Video Summarizer + Chat/Quiz



In [4]:
!pip install -q flask flask-cors youtube-transcript-api "transformers<5" torch pyngrok sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3HKXHklXW5h8VnZMAWBTolh1ZSv_3eYxbRDfeZGgKLxSgnyxX"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

## 2) Front end files


In [6]:
import os
os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

In [7]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>YT Summarizer</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
  <div class="container">
    <header>
      <h1>Video Summarizer</h1>
      <p class="subtitle">الصقي رابط أي فيديو يوتيوب وهياخدلك ملخص، وتقدري تسأليه أو تعملي كويز</p>
    </header>

    <div class="input-card">
      <input type="text" id="urlInput" placeholder="https://www.youtube.com/watch?v=..." />
      <select id="langSelect">
        <option value="en">English</option>
        <option value="ar">العربية</option>
      </select>
      <button id="summarizeBtn">لخّص الفيديو</button>
    </div>

    <div id="status" class="status hidden"></div>

    <div id="appBody" class="hidden">
      <div class="tabs">
        <button class="tab-btn active" data-tab="summary">الملخص</button>
        <button class="tab-btn" data-tab="chat">اسألي عن الفيديو</button>
        <button class="tab-btn" data-tab="quiz">كويز</button>
      </div>

      <div id="tab-summary" class="tab-content active">
        <div class="result-card">
          <div class="meta">
            <span id="videoIdBadge"></span>
            <span id="lengthBadge"></span>
          </div>
          <h2>الملخص</h2>
          <p id="summaryText"></p>
        </div>
      </div>

      <div id="tab-chat" class="tab-content">
        <div class="result-card chat-card">
          <div id="chatMessages" class="chat-messages"></div>
          <div class="chat-input-row">
            <input id="chatInput" placeholder="اكتبي سؤالك بالإنجليزي عشان نتيجة أدق" />
            <button id="chatSendBtn">إرسال</button>
          </div>
        </div>
      </div>

      <div id="tab-quiz" class="tab-content">
        <div class="result-card">
          <div class="quiz-header">
            <button id="generateQuizBtn">ولّد كويز (5 أسئلة)</button>
          </div>
          <div id="quizContainer"></div>
        </div>
      </div>
    </div>
  </div>

  <script src="{{ url_for('static', filename='script.js') }}"></script>
</body>
</html>

Writing templates/index.html


In [8]:
%%writefile static/style.css
* { box-sizing: border-box; margin: 0; padding: 0; }

body {
  font-family: "Segoe UI", Tahoma, sans-serif;
  background: linear-gradient(135deg, #1f1147 0%, #4a1a63 100%);
  min-height: 100vh;
  display: flex;
  justify-content: center;
  padding: 40px 16px;
  color: #fff;
}

.container { max-width: 760px; width: 100%; }

header { text-align: center; margin-bottom: 32px; }
header h1 { font-size: 2rem; margin-bottom: 8px; }
.subtitle { color: #d9c9f0; }

.input-card {
  background: rgba(255,255,255,0.08);
  border: 1px solid rgba(255,255,255,0.15);
  border-radius: 16px;
  padding: 20px;
  display: flex;
  gap: 10px;
  flex-wrap: wrap;
  backdrop-filter: blur(6px);
}

#urlInput { flex: 1; min-width: 200px; padding: 12px 14px; border-radius: 10px; border: none; font-size: 1rem; }
#langSelect { padding: 12px 10px; border-radius: 10px; border: none; font-size: 1rem; }

#summarizeBtn, #generateQuizBtn, #chatSendBtn {
  padding: 12px 22px; border: none; border-radius: 10px;
  background: #7c3aed; color: #fff; font-weight: bold; cursor: pointer;
  transition: background .2s;
}
#summarizeBtn:hover, #generateQuizBtn:hover, #chatSendBtn:hover { background: #6d28d9; }
#summarizeBtn:disabled, #generateQuizBtn:disabled, #chatSendBtn:disabled { background: #555; cursor: not-allowed; }

.status { margin-top: 18px; padding: 14px; border-radius: 10px; text-align: center; }
.status.loading { background: rgba(124,58,237,0.25); }
.status.error { background: rgba(220,38,38,0.3); }
.hidden { display: none; }

.tabs { display: flex; gap: 8px; margin-top: 28px; margin-bottom: 4px; }
.tab-btn {
  flex: 1; padding: 10px; border: none; border-radius: 10px 10px 0 0;
  background: rgba(255,255,255,0.06); color: #d9c9f0; cursor: pointer; font-weight: bold;
}
.tab-btn.active { background: rgba(255,255,255,0.15); color: #fff; }

.tab-content { display: none; }
.tab-content.active { display: block; }

.result-card {
  margin-top: 0;
  background: rgba(255,255,255,0.08);
  border: 1px solid rgba(255,255,255,0.15);
  border-radius: 0 0 16px 16px;
  padding: 24px;
}

.meta { display: flex; gap: 10px; margin-bottom: 14px; font-size: 0.85rem; flex-wrap: wrap; }
.meta span { background: rgba(255,255,255,0.15); padding: 4px 10px; border-radius: 999px; }

.result-card h2 { margin-bottom: 10px; color: #d9c9f0; }
.result-card p { line-height: 1.8; font-size: 1.05rem; }

/* Chat */
.chat-card { display: flex; flex-direction: column; height: 420px; }
.chat-messages { flex: 1; overflow-y: auto; display: flex; flex-direction: column; gap: 10px; margin-bottom: 12px; }
.msg { padding: 10px 14px; border-radius: 12px; max-width: 85%; line-height: 1.6; }
.msg.user { background: #7c3aed; align-self: flex-start; }
.msg.bot { background: rgba(255,255,255,0.12); align-self: flex-end; }
.chat-input-row { display: flex; gap: 8px; }
#chatInput { flex: 1; padding: 10px 14px; border-radius: 10px; border: none; font-size: 1rem; }

/* Quiz */
.quiz-header { margin-bottom: 16px; }
.quiz-question { margin-bottom: 20px; padding-bottom: 16px; border-bottom: 1px solid rgba(255,255,255,0.1); }
.quiz-question h3 { margin-bottom: 10px; }
.quiz-option {
  display: block; width: 100%; text-align: right; padding: 10px 14px; margin-bottom: 8px;
  border-radius: 8px; border: 1px solid rgba(255,255,255,0.2); background: rgba(255,255,255,0.05);
  color: #fff; cursor: pointer; font-size: 0.95rem;
}
.quiz-option:hover { background: rgba(255,255,255,0.12); }
.quiz-option.correct { background: rgba(34,197,94,0.4); border-color: #22c55e; }
.quiz-option.wrong { background: rgba(220,38,38,0.4); border-color: #dc2626; }
.quiz-option:disabled { cursor: not-allowed; }

Writing static/style.css


In [9]:
%%writefile static/script.js
const btn = document.getElementById("summarizeBtn");
const urlInput = document.getElementById("urlInput");
const langSelect = document.getElementById("langSelect");
const statusEl = document.getElementById("status");
const appBody = document.getElementById("appBody");
const summaryText = document.getElementById("summaryText");
const videoIdBadge = document.getElementById("videoIdBadge");
const lengthBadge = document.getElementById("lengthBadge");

const chatMessages = document.getElementById("chatMessages");
const chatInput = document.getElementById("chatInput");
const chatSendBtn = document.getElementById("chatSendBtn");

const generateQuizBtn = document.getElementById("generateQuizBtn");
const quizContainer = document.getElementById("quizContainer");

let currentVideoId = null;

function showStatus(message, type) {
  statusEl.textContent = message;
  statusEl.className = `status ${type}`;
}
function hideStatus() { statusEl.className = "status hidden"; }

document.querySelectorAll(".tab-btn").forEach(tabBtn => {
  tabBtn.addEventListener("click", () => {
    document.querySelectorAll(".tab-btn").forEach(b => b.classList.remove("active"));
    document.querySelectorAll(".tab-content").forEach(c => c.classList.remove("active"));
    tabBtn.classList.add("active");
    document.getElementById(`tab-${tabBtn.dataset.tab}`).classList.add("active");
  });
});

async function summarize() {
  const url = urlInput.value.trim();
  const lang = langSelect.value;
  if (!url) { showStatus("من فضلك ادخل رابط فيديو", "error"); return; }

  appBody.classList.add("hidden");
  btn.disabled = true;
  showStatus("بيجيب الترانسكريبت ويلخص... ممكن ياخد شوية وقت", "loading");

  try {
    const res = await fetch("/api/summarize", {
      method: "POST",
      headers: { "Content-Type": "application/json", "ngrok-skip-browser-warning": "true" },
      body: JSON.stringify({ url, lang }),
    });
    const data = await res.json();
    if (!res.ok) { showStatus(`${data.error || "حصل خطأ"}`, "error"); return; }

    hideStatus();
    currentVideoId = data.video_id;
    videoIdBadge.textContent = `Video: ${data.video_id}`;
    lengthBadge.textContent = `${data.transcript_length_chars} حرف`;
    summaryText.textContent = data.summary;
    appBody.classList.remove("hidden");

    chatMessages.innerHTML = "";
    quizContainer.innerHTML = "";
  } catch (err) {
    showStatus(`فشل الاتصال بالسيرفر: ${err.message}`, "error");
  } finally {
    btn.disabled = false;
  }
}

btn.addEventListener("click", summarize);
urlInput.addEventListener("keydown", (e) => { if (e.key === "Enter") summarize(); });

function addMessage(text, sender) {
  const div = document.createElement("div");
  div.className = `msg ${sender}`;
  div.textContent = text;
  chatMessages.appendChild(div);
  chatMessages.scrollTop = chatMessages.scrollHeight;
}

async function sendChat() {
  const question = chatInput.value.trim();
  if (!question || !currentVideoId) return;

  addMessage(question, "user");
  chatInput.value = "";
  chatSendBtn.disabled = true;
  addMessage("...", "bot");

  try {
    const res = await fetch("/api/chat", {
      method: "POST",
      headers: { "Content-Type": "application/json", "ngrok-skip-browser-warning": "true" },
      body: JSON.stringify({ video_id: currentVideoId, question }),
    });
    const data = await res.json();
    chatMessages.lastChild.remove();
    if (!res.ok) { addMessage(`خطأ: ${data.error}`, "bot"); return; }
    addMessage(data.answer, "bot");
  } catch (err) {
    chatMessages.lastChild.remove();
    addMessage(`فشل الاتصال: ${err.message}`, "bot");
  } finally {
    chatSendBtn.disabled = false;
  }
}

chatSendBtn.addEventListener("click", sendChat);
chatInput.addEventListener("keydown", (e) => { if (e.key === "Enter") sendChat(); });

async function generateQuiz() {
  if (!currentVideoId) return;
  generateQuizBtn.disabled = true;
  quizContainer.innerHTML = "<p>بيولّد الأسئلة... (ممكن ياخد دقيقة، بيشتغل خطوة خطوة)</p>";

  try {
    const res = await fetch("/api/quiz", {
      method: "POST",
      headers: { "Content-Type": "application/json", "ngrok-skip-browser-warning": "true" },
      body: JSON.stringify({ video_id: currentVideoId, num_questions: 5 }),
    });
    const data = await res.json();
    if (!res.ok) { quizContainer.innerHTML = `<p>خطأ: ${data.error}</p>`; return; }
    renderQuiz(data.questions);
  } catch (err) {
    quizContainer.innerHTML = `<p>فشل الاتصال: ${err.message}</p>`;
  } finally {
    generateQuizBtn.disabled = false;
  }
}

function renderQuiz(questions) {
  quizContainer.innerHTML = "";
  questions.forEach((q, qIdx) => {
    const qDiv = document.createElement("div");
    qDiv.className = "quiz-question";
    qDiv.innerHTML = `<h3>${qIdx + 1}. ${q.question}</h3>`;

    q.options.forEach((opt, optIdx) => {
      const optBtn = document.createElement("button");
      optBtn.className = "quiz-option";
      optBtn.textContent = opt;
      optBtn.addEventListener("click", () => {
        const allOpts = qDiv.querySelectorAll(".quiz-option");
        allOpts.forEach(o => o.disabled = true);
        if (optIdx === q.correct_index) {
          optBtn.classList.add("correct");
        } else {
          optBtn.classList.add("wrong");
          allOpts[q.correct_index].classList.add("correct");
        }
      });
      qDiv.appendChild(optBtn);
    });

    quizContainer.appendChild(qDiv);
  });
}

generateQuizBtn.addEventListener("click", generateQuiz);

Writing static/script.js


## 3) load models


In [10]:
import re
from urllib.parse import urlparse, parse_qs
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| لو عايزة GPU: Runtime > Change runtime type > GPU")

# موديل التلخيص
SUM_MODEL_NAME = "facebook/bart-large-cnn"
sum_tokenizer = AutoTokenizer.from_pretrained(SUM_MODEL_NAME)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(SUM_MODEL_NAME).to(device)

# موديل الشات/الكويز (مجاني، instruction-tuned)
QA_MODEL_NAME = "google/flan-t5-large"
qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
qa_model = AutoModelForSeq2SeqLM.from_pretrained(QA_MODEL_NAME).to(device)

print("الموديلات اتحملت.")

Device: cpu | لو عايزة GPU: Runtime > Change runtime type > GPU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

الموديلات اتحملت.


## 4) summrization

In [11]:
def extract_video_id(url: str) -> str:
    parsed = urlparse(url)
    if parsed.hostname in ("youtu.be",):
        return parsed.path.lstrip("/")
    if parsed.hostname and "youtube.com" in parsed.hostname:
        qs = parse_qs(parsed.query)
        if "v" in qs:
            return qs["v"][0]
        match = re.search(r"/shorts/([A-Za-z0-9_-]+)", parsed.path)
        if match:
            return match.group(1)
    raise ValueError(f"Couldn\'t extract video id from: {url}")


def get_transcript_text(video_id: str, lang: str = "en") -> str:
    api = YouTubeTranscriptApi()
    try:
        fetched = api.fetch(video_id, languages=[lang])
    except (TranscriptsDisabled, NoTranscriptFound):
        fetched = api.fetch(video_id)
    return "\n".join(snippet.text for snippet in fetched)


def summarize_long_text(text: str, chunk_max_length=130, chunk_min_length=30) -> str:
    tokens = sum_tokenizer(text, return_tensors=None, return_overflowing_tokens=True,
                            truncation=True, max_length=1024)
    summaries = []
    for chunk_ids in tokens["input_ids"]:
        input_tensor = torch.tensor([chunk_ids]).to(device)
        summary_ids = sum_model.generate(input_tensor, max_length=chunk_max_length,
                                          min_length=chunk_min_length, num_beams=4, early_stopping=True)
        summaries.append(sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True))

    combined = " ".join(summaries)
    if len(summaries) > 1:
        final_tokens = sum_tokenizer(combined, truncation=True, max_length=1024)
        input_tensor = torch.tensor([final_tokens["input_ids"]]).to(device)
        final_ids = sum_model.generate(input_tensor, max_length=150, min_length=50,
                                        num_beams=4, early_stopping=True)
        return sum_tokenizer.decode(final_ids[0], skip_special_tokens=True)
    return combined

## 5)chatbot & Quiz


In [12]:
import random

MAX_CONTEXT_CHARS = 3000  # flan-t5 بياخد context محدود (512 token تقريبًا)


def qa_generate(prompt: str, max_new_tokens: int = 60) -> str:
    inputs = qa_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = qa_model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4, early_stopping=True)
    return qa_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def answer_question(transcript: str, question: str) -> str:
    context = transcript[:MAX_CONTEXT_CHARS]
    prompt = f"Answer the question based on the context.\n\nContext: {context}\n\nQuestion: {question}\nAnswer:"
    return qa_generate(prompt, max_new_tokens=80)


def generate_one_question(context: str) -> dict:
    q_prompt = f"Read the context and write one specific quiz question about a fact mentioned in it.\n\nContext: {context}\n\nQuestion:"
    question = qa_generate(q_prompt, max_new_tokens=40)

    a_prompt = f"Context: {context}\n\nQuestion: {question}\nAnswer in 1-4 words:"
    correct_answer = qa_generate(a_prompt, max_new_tokens=15)

    d_prompt = (
        f"Question: {question}\nCorrect answer: {correct_answer}\n"
        f"Write 3 short incorrect but plausible answers, separated by ' | '. Do not repeat the correct answer.\nWrong answers:"
    )
    distractors_raw = qa_generate(d_prompt, max_new_tokens=40)
    distractors = [d.strip(" .") for d in re.split(r"\||,|;", distractors_raw) if d.strip()]
    distractors = [d for d in distractors if d.lower() != correct_answer.lower()][:3]

    # fallback لو الموديل مطلعش 3 إجابات غلط كفاية
    generic_fallbacks = ["None of the above", "Not mentioned in the video", "All of the above"]
    while len(distractors) < 3:
        distractors.append(generic_fallbacks[len(distractors) % len(generic_fallbacks)])

    options = distractors[:3] + [correct_answer]
    random.shuffle(options)
    correct_index = options.index(correct_answer)

    return {"question": question, "options": options, "correct_index": correct_index}


def generate_quiz(transcript: str, num_questions: int = 5) -> list:
    context = transcript[:MAX_CONTEXT_CHARS]
    questions = []
    seen = set()
    attempts = 0
    while len(questions) < num_questions and attempts < num_questions * 3:
        attempts += 1
        q = generate_one_question(context)
        if q["question"] and q["question"] not in seen:
            seen.add(q["question"])
            questions.append(q)
    return questions

In [13]:
from flask import Flask, request, jsonify, render_template
from flask_cors import CORS
from pyngrok import ngrok
import threading

app = Flask(__name__)
CORS(app)

TRANSCRIPT_CACHE = {}  # video_id -> transcript text


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/api/summarize", methods=["POST"])
def summarize():
    data = request.get_json(silent=True) or {}
    url = data.get("url", "").strip()
    lang = data.get("lang", "en").strip() or "en"

    if not url:
        return jsonify({"error": "من فضلك ادخل رابط الفيديو"}), 400
    try:
        video_id = extract_video_id(url)
    except ValueError as e:
        return jsonify({"error": str(e)}), 400
    try:
        transcript = get_transcript_text(video_id, lang=lang)
    except (TranscriptsDisabled, NoTranscriptFound):
        return jsonify({"error": "الفيديو ده مفهوش ترانسكريبت/كابشنز متاحة"}), 404
    except Exception as e:
        return jsonify({"error": f"مشكلة في جلب الترانسكريبت: {e}"}), 500
    if not transcript.strip():
        return jsonify({"error": "الترانسكريبت طلع فاضي"}), 404

    TRANSCRIPT_CACHE[video_id] = transcript

    try:
        summary = summarize_long_text(transcript)
    except Exception as e:
        return jsonify({"error": f"مشكلة في التلخيص: {e}"}), 500

    return jsonify({"video_id": video_id, "transcript_length_chars": len(transcript), "summary": summary})


@app.route("/api/chat", methods=["POST"])
def chat():
    data = request.get_json(silent=True) or {}
    video_id = data.get("video_id", "").strip()
    question = data.get("question", "").strip()

    if not video_id or video_id not in TRANSCRIPT_CACHE:
        return jsonify({"error": "لازم تلخصي الفيديو الأول"}), 400
    if not question:
        return jsonify({"error": "اكتبي سؤال"}), 400

    try:
        answer = answer_question(TRANSCRIPT_CACHE[video_id], question)
    except Exception as e:
        return jsonify({"error": f"مشكلة في الرد: {e}"}), 500

    return jsonify({"answer": answer})


@app.route("/api/quiz", methods=["POST"])
def quiz():
    data = request.get_json(silent=True) or {}
    video_id = data.get("video_id", "").strip()
    num_questions = int(data.get("num_questions", 5))

    if not video_id or video_id not in TRANSCRIPT_CACHE:
        return jsonify({"error": "لازم تلخصي الفيديو الأول"}), 400

    try:
        questions = generate_quiz(TRANSCRIPT_CACHE[video_id], num_questions)
    except Exception as e:
        return jsonify({"error": f"مشكلة في توليد الكويز: {e}"}), 500

    if not questions:
        return jsonify({"error": "معرفتش أولّد أسئلة كفاية من المحتوى ده"}), 500

    return jsonify({"questions": questions})


ngrok.kill()  # يقفل أي نفق ngrok قديم عالق قبل ما يفتح واحد جديد
public_url = ngrok.connect(5000)
print("رابط الموقع العام:", public_url)

threading.Thread(target=lambda: app.run(port=5000)).start()

رابط الموقع العام: NgrokTunnel: "https://sculptor-smile-subsoil.ngrok-free.dev" -> "http://localhost:5000"
